# TF-IDF

- 문서에서 특정 단어가 얼마나 중요한지를 수치로 표현하는 방법

## TF
- 특정 단어가 문서에 얼마나 자주 등장하는지를 의미하는 수치
- 특정단어가 등장한 횟수 / 문서의 전체 단어수

## DTM
- 단어의 등장 빈도를 문서별로 표현하는 방식

## IDF
- 단어가 다른 문서들에서 얼마나 드물게 나타나고 있는지를 측정한다.
- 어떤 특정 단어가 전체 문서에서 흔하게 등장하면, 해당 언어는 특이성이 낮아 중요도가 낮다고 볼 수 있다.
- 반대로 특정 주제에만 나타나는 단어일수록 그 문서에서의 중요성이 높아지게 된다.


# BM25

- TF-IDF(총 문서의 개수와 단어의 등장 빈도수를 고려하는 알고리즘) 를 기반으로 문서의 길이까지 고려해서 문서와 쿼리 간 관련성까지 평가한다.
- 일정 빈도 이상에서는 관련성을 천천히 증가시키는 포화 효과를 통해 보정 작업을 거친다.
- 길이가 긴 문서인 경우 짧은 문서에 비해 단어가 많기 때문에 상대적으로 TF가 높을 수 있으므로, 문서의 길이에 대한 보정식도 적용한다.
- 주로 검색 엔진이나 정보 검색 시스템에서 정밀한 결과를 제공하여 사용자에게 더욱 적절한 문서를 추천 할 수 있다.


# BM25 Score 구하기 실습

In [13]:
# 1. 관련 라이브러리 불러오기

from rank_bm25 import BM25Okapi
import pandas as pd
from kiwipiepy import Kiwi

In [14]:
# 2. 문서 준비
documents = [
    '고양이와 강아지', # 문서 1
    '우리 강아지가 다른 강아지를 쫓아갔어', # 문서 2
    '강아지는 창밖을 봐' # 문서 3
]

In [15]:
# 3. 한국어 토근화를 위한 함수 작성
kiwi = Kiwi() # 한국어 형태소 분석기

def tokenize_korean_text(text): # 문서 내용 토근화하는 함수
    # 리스트 컴프리헨션(내포)
    # [최종 결과 for 변수명 in 반복가능한 객체]
    return [token.form for token in kiwi.tokenize(text)]

In [5]:
tokenize_korean_text('우리는 RAG 시스템을 공부합니다.')

['우리', '는', 'RAG', '시스템', '을', '공부', '하', 'ᆸ니다', '.']

In [6]:
# 4. 각 문서를 토근화 (단어 단위로 분할)
tokenize_documents = []
for doc in documents:
    tokenize_documents.append(tokenize_korean_text(doc))
    

In [7]:
print(tokenize_documents)

[['고양이', '와', '강아지'], ['우리', '강아지', '가', '다른', '강아지', '를', '쫓아가', '었', '어'], ['강아지', '는', '창', '밖', '을', '보', '어']]


In [9]:
# 5. 불필요한 불용어 제거
def delete_particles(tokenized_docs: list,
                     particles: list=['와','가','를','는','을']):
    for sentence_list in tokenized_docs:
        for token in sentence_list:
            if token in particles: # 토큰에 담긴 값이 불용어리스트에 포함되었다면
                sentence_list.remove(token) # 값을 지운다.
    return tokenized_docs

In [10]:
print(delete_particles(tokenize_documents)) # 함수 호출 후 프린트

[['고양이', '강아지'], ['우리', '강아지', '다른', '강아지', '쫓아가', '었', '어'], ['강아지', '창', '밖', '보', '어']]


In [11]:
# 6. BM25 알고리즘을 사용한 객체 생성

bm25 = BM25Okapi(tokenize_documents, k1=1.5,b=0.75)

In [12]:
# 7. 문서의 모든 단어를 추출하여 고유 단어 목록 생성

terms = [word for doc in tokenize_documents for word in doc] # 모든 단어를 추출
terms


['고양이',
 '강아지',
 '우리',
 '강아지',
 '다른',
 '강아지',
 '쫓아가',
 '었',
 '어',
 '강아지',
 '창',
 '밖',
 '보',
 '어']

In [14]:
terms = set(terms) # 세트 형태로 변환 후 다시 저장
terms

{'강아지', '고양이', '다른', '밖', '보', '어', '었', '우리', '쫓아가', '창'}

### 파이썬의 컬렉션 자료형(복수)
- 리스트, 튜플, 딕셔너리, 세트(중복제거, 순서가 없다.)

In [16]:
terms = list(terms)
terms

['다른', '창', '우리', '보', '강아지', '어', '쫓아가', '밖', '고양이', '었']

In [17]:
# 8. 각 문서에 대해 BM25 점수를 계산하여 저장

bm25_scores = []
for term in terms:
    term_scores = bm25.get_scores([term]) # BM25 점수 구하기
    bm25_scores.append(term_scores) # 결과를 리스트에 모은다

In [18]:
bm25_scores

[array([0.        , 0.41700051, 0.        ]),
 array([0.        , 0.        , 0.49491756]),
 array([0.        , 0.41700051, 0.        ]),
 array([0.        , 0.        , 0.49491756]),
 array([0.05485137, 0.05014982, 0.0394778 ]),
 array([0.        , 0.03326264, 0.0394778 ]),
 array([0.        , 0.41700051, 0.        ]),
 array([0.        , 0.        , 0.49491756]),
 array([0.68764988, 0.        , 0.        ]),
 array([0.        , 0.41700051, 0.        ])]

In [19]:
# 9. 결과를 데이터 프레임으로 변환

bm25_df = pd.DataFrame(bm25_scores, index=terms, columns=['문서1','문서2','문서3'])

In [20]:
bm25_df

,문서1,문서2,문서3
다른,0.000000,0.417001,0.000000
창,0.000000,0.000000,0.494918
우리,0.000000,0.417001,0.000000
보,0.000000,0.000000,0.494918
강아지,0.054851,0.050150,0.039478
어,0.000000,0.033263,0.039478
쫓아가,0.000000,0.417001,0.000000
밖,0.000000,0.000000,0.494918
고양이,0.687650,0.000000,0.000000
었,0.000000,0.417001,0.000000


BM25를 활요한 RAG system - 라마인덱스

In [21]:
# 1. 사용할 문서 불러오기
from llama_index.core import SimpleDirectoryReader
documents = SimpleDirectoryReader('./data').load_data()
print(f'문서의 총 개수 : {len(documents)}')

문서의 총 개수 : 10


## KeyworedTableIndex
- 문서를 기반으로 키워드와 노드의 매핑을 생성하는 인덱스 클래스

In [16]:
# 4. 저장된 인덱스 불러오기
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir='./index/bm25_index')
index = load_index_from_storage(storage_context)

In [17]:
index

In [18]:
# 5. BM25 검색기 객체 생성 및 쿼리 검색

from llama_index.retrievers.bm25 import BM25Retriever

bm25_retriever = BM25Retriever.from_defaults(
    index=index,  # bm25를 적용할 인덱스
    similarity_top_k=2, # 유사도 기반 상위 k개 결과 반환
    tokenizer=tokenize_korean_text, # kiwi 토크나이저 사용
)

The tokenizer parameter is deprecated and will be removed in a future release. Use a stemmer from PyStemmer instead.


In [20]:
# 6. 쿼리 검색

from llama_index.core.response.notebook_utils import display_source_node

query = '변동금리'
retrieverd_nodes = bm25_retriever.retrieve(query)
for node in retrieverd_nodes:
    display_source_node(node, source_length=10000)

ModuleNotFoundError: No module named 'matplotlib'